# Träger-Experiment WASD — Messlauf

Vorregistrierter Lauf zur Frage, ob die Zeichenfolge `WASD` im Tokenraum von
`EleutherAI/pythia-1.4b` über eine gewöhnliche lexikalische Assoziation hinaus trägt.

**Das Design in einem Satz.** `" WASD"` zerfällt in genau zwei Token, `['ĠWAS', 'D']`.
Alle Varianten `WAS?` zerfallen ebenso — zwischen Ziel und Kontrolle unterscheidet sich
genau ein Token, und zwar das, dem die Hypothese ihre Bedeutung zuschreibt.

**Die Entscheidungsregel steht vor dem Lauf fest** und verlangt beides:

1. *Beobachtung* — das Ziel hebt das Bewegungsvokabular um mindestens **0.5 nats**
   gegenüber der **besten** Kontrolle, nicht gegenüber deren Mittel.
2. *Kausalität* — ein Eingriff an der Position des letzten Tokens stellt mindestens die
   **Hälfte** des Effekts wieder her, an mindestens **zwei benachbarten** Blöcken,
   während derselbe Eingriff mit Kontrollquelle höchstens **ein Fünftel** überträgt.

Ein einzelner positiver Teil zählt nicht.

**Die drei Zusatzkontrollen** schließen die Lücken, die die Buchstabenfamilie offenlässt:

| Kontrolle | Zerlegung | Wozu |
|---|---|---|
| `FORD` | `ĠFOR` `D` | **dasselbe Endtoken** (ID 37) wie das Ziel, anderer Stamm. Trüge allein der Buchstabe `D`, müsste FORD es zeigen. |
| `NOTA` | `ĠNOT` `A` | strukturgleich und im Korpus fast gleich häufig (14 286 gegen 13 870). Die eigentliche Nulllinie. |
| `ESDF` | `ĠE` `SD` `F` | echte alternative Tastenbelegung, aber **nicht** strukturgleich und seltener. Geht **nicht** in die Regel ein, wird getrennt berichtet. |

Der Lauf schreibt am Ende `FINAL_REPORT.md`, `FINAL_SUMMARY.json`, `QA.json`,
`measurement_evidence.json` und die Rohtabellen, dazu die Marke `COMPLETE`.


## 1 · Konfiguration

In [ ]:
# Alles, was den Lauf bestimmt, steht in dieser Zelle.
MODELL_ID   = "EleutherAI/pythia-1.4b"
REVISION    = "step98000"
PRAEZISION  = "float32"
BATCHGROESSE = 1

ZIEL            = "WASD"
ZIEL_ZERLEGUNG  = ("\u0120WAS", "D")
REIHENFOLGE_KONTROLLE = "ASDW"

# (Text, Rolle) - die Rollen steuern, was in die Hauptregel eingeht.
# ROLLE_GETRENNT steht einmal hier und wird ueberall sonst nur noch referenziert,
# damit die Ausnahme nicht an drei Stellen getippt und irgendwann uneinheitlich wird.
ROLLE_GETRENNT = "tastenkontrolle_unabgeglichen"
ZUSATZKONTROLLEN = (
    ("FORD", "endtokenkontrolle"),
    ("NOTA", "frequenzkontrolle"),
    ("ESDF", ROLLE_GETRENNT),   # NICHT in der Hauptregel
)
ZUSATZROLLEN = frozenset(rolle for _, rolle in ZUSATZKONTROLLEN)

# Die fuenf Paare des Eingriffsteils - vorab festgelegt, nicht nach den Grundlinien
# ausgewaehlt. Eine Auswahl nach dem Ergebnis waere zirkulaer.
EINGRIFF_KONTROLLEN = ("FORD", "NOTA", "WASF", "WASR", "WASZ")

MINDEST_KONTROLLEN   = 15
MINDEST_VORSPRUNG    = 0.5    # nats, gegen die beste Kontrolle
KONTROLL_QUANTIL     = 0.90   # vorab benannte Zweitfassung
MINDEST_KAUSALANTEIL = 0.50
HOECHSTES_LECK       = 0.20
MINDEST_SCHICHTEN    = 2

# Abbruch, wenn mehr als dieser Anteil der Transferzeilen keinen brauchbaren
# Nenner hat. Im Vorlauf betraf das 144 von 672 Zeilen.
HOECHSTER_ANTEIL_ZU_KLEINER_ABSTAND = 0.20
MINDEST_ABSTAND_NATS = 0.01   # |Spenderwert - Empfaengerwert| darunter: unbrauchbar

# Das Residualtor prueft eine Rekonstruktion, keine Identitaet: resid_in + attn + mlp
# summiert in anderer Reihenfolge als der Block selbst, und float32 rundet dabei
# anders. Verlangt wird deshalb der Rundungsboden, nicht die Null. Eine strukturell
# falsche Zerlegung - etwa ein sequentielles statt parallelen Residuum - laege um
# viele Groessenordnungen darueber und faellt hier sicher durch.
HOECHSTER_RESIDUALFEHLER_REL = 1e-4

AUSGABE = "./wasd_traeger_lauf"

In [ ]:
# Jedes Wort muss ein EINZELNES Token sein, sonst misst die Sonde das erste
# Stueck statt des Wortes. " strafe" (['\u0120stra','fe']) und " senate"
# (['\u0120sen','ate']) erfuellen das nicht und wurden vor der Messung ersetzt.
MESSVOKABULAR = {
    "bewegung":  (" move", " walk", " sprint", " forward", " backward"),
    "tastatur":  (" keys", " key", " keyboard", " bind", " controls"),
    "gegenfeld": (" weather", " orbit", " proof", " recipe", " parliament"),
}

# Am Tokenizer von pythia-1.4b abgelesen. Weicht der geladene Tokenizer ab, ist es
# nicht dieses Modell - und keine Zahl des Laufs waere vergleichbar.
MESSVOKABULAR_IDS = {
    "bewegung":  (2118, 2940, 29644, 3579, 19265),
    "tastatur":  (10149, 2234, 15487, 8980, 5760),
    "gegenfeld": (8588, 12801, 4737, 13612, 16005),
}

# 24 Schablonen. Jede endet mit dem Platzhalter - gemessen wird die Verteilung des
# unmittelbar folgenden Tokens.
SCHABLONEN = (
    "In the options menu you can rebind the movement keys. By default the game uses {ziel}",
    "Controls: the player character is moved with {ziel}",
    "Press the standard PC layout keys {ziel}",
    "The tutorial explains that you should hold {ziel}",
    "Most first person shooters map movement to {ziel}",
    "He kept his left hand on {ziel}",
    "The config file remaps the arrow keys to {ziel}",
    "Movement is bound to the four keys {ziel}",
    "You walk around the level using {ziel}",
    "The default binding for movement is {ziel}",
    "Setting: movement keys {ziel}",
    "For strafing and walking forward, use {ziel}",
    "The manual says the character responds to {ziel}",
    "Remap the directional input from the arrows to {ziel}",
    "Her fingers rested on {ziel}",
    "Beginners are told to learn {ziel}",
    "The keybind menu lists movement under {ziel}",
    "Standard PC controls put movement on {ziel}",
    "To move the avatar, press {ziel}",
    "The readme documents the movement cluster {ziel}",
    "Navigation in the editor is bound to {ziel}",
    "Players consistently prefer {ziel}",
    "The input handler reads the keys {ziel}",
    "Forward, back, left and right are mapped to {ziel}",
)
print(f"{len(SCHABLONEN)} Schablonen")
assert all(s.endswith("{ziel}") for s in SCHABLONEN), "jede Schablone muss mit dem Ziel enden"

## 2 · Vorflug

Fünf Prüfungen, bevor Gewichte geladen werden. Schlägt eine fehl, startet der Lauf nicht.

In [ ]:
import hashlib, json, os, platform, sys, time
from dataclasses import dataclass, asdict

import torch, transformers, tokenizers
from transformers import AutoModelForCausalLM, AutoTokenizer

os.makedirs(AUSGABE, exist_ok=True)

# Determinismus. Batchgroesse eins und FP32 sind Teil der Vorregistrierung.
torch.use_deterministic_algorithms(True, warn_only=True)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.manual_seed(60482)

VORFLUG = {
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "tokenizers": tokenizers.__version__,
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "tf32_matmul": torch.backends.cuda.matmul.allow_tf32,
}
for k, v in VORFLUG.items():
    print(f"  {k:16s} {v}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODELL_ID, revision=REVISION)

# Pruefsumme des Tokenizers. Eine stillschweigend geaenderte Tokenisierung wuerde
# den gesamten Vergleich entwerten, ohne dass es auffiele.
_tokdatei = os.path.join(tokenizer.name_or_path, "tokenizer.json")
if not os.path.exists(_tokdatei):
    from huggingface_hub import hf_hub_download
    _tokdatei = hf_hub_download(MODELL_ID, "tokenizer.json", revision=REVISION)
TOKENIZER_SHA256 = hashlib.sha256(open(_tokdatei, "rb").read()).hexdigest()
VORFLUG["tokenizer_sha256"] = TOKENIZER_SHA256
print("tokenizer sha256:", TOKENIZER_SHA256)

In [ ]:
@dataclass(frozen=True)
class Variante:
    text: str
    rolle: str
    zerlegung: tuple
    token_ids: tuple

    @property
    def letztes_token(self):
        return self.zerlegung[-1]


def zerlege(text):
    ids = tokenizer.encode(" " + text)
    return tuple(tokenizer.convert_ids_to_tokens(ids)), tuple(ids)


def baue_varianten():
    v = []
    def erfasse(text, rolle):
        z, ids = zerlege(text)
        v.append(Variante(text, rolle, z, ids))
    erfasse(ZIEL, "ziel")
    stamm = ZIEL[:-1]
    for b in "ABCDEFGHIJKLMNOPQRSTUVWXYZ":
        if b != ZIEL[-1]:
            erfasse(stamm + b, "buchstabenkontrolle")
    erfasse(REIHENFOLGE_KONTROLLE, "reihenfolgekontrolle")
    for text, rolle in ZUSATZKONTROLLEN:
        erfasse(text, rolle)
    return v


VARIANTEN = baue_varianten()

def strukturgleich(va):
    return (va.rolle == "buchstabenkontrolle"
            and len(va.zerlegung) == len(ZIEL_ZERLEGUNG)
            and va.zerlegung[0] == ZIEL_ZERLEGUNG[0])

BRAUCHBAR  = [v for v in VARIANTEN if strukturgleich(v)]
VERWORFEN  = sorted({v.text for v in VARIANTEN if v.rolle == "buchstabenkontrolle"}
                    - {v.text for v in BRAUCHBAR})
ZIELVAR    = next(v for v in VARIANTEN if v.rolle == "ziel")

maengel = []
if ZIELVAR.zerlegung != ZIEL_ZERLEGUNG:
    maengel.append(f"Ziel zerfaellt in {list(ZIELVAR.zerlegung)}, erwartet {list(ZIEL_ZERLEGUNG)}")
if len(BRAUCHBAR) < MINDEST_KONTROLLEN:
    maengel.append(f"nur {len(BRAUCHBAR)} strukturgleiche Kontrollen")
for v in VARIANTEN:
    # Die unabgeglichene Tastenkontrolle darf die Struktur verletzen - deshalb ist
    # sie ja aus der Hauptregel heraus. Alle anderen Zusatzkontrollen duerfen nicht.
    if v.rolle in ZUSATZROLLEN and v.rolle != ROLLE_GETRENNT and len(v.zerlegung) != len(ZIEL_ZERLEGUNG):
        maengel.append(f"Zusatzkontrolle {v.text}: {list(v.zerlegung)}")

# Die Eingriffspaare stehen vorab fest; sie muessen auch wirklich gemessen werden.
_texte = {v.text for v in VARIANTEN}
_fehlend = [k for k in EINGRIFF_KONTROLLEN if k not in _texte]
assert not _fehlend, f"Eingriffskontrollen ohne Variante: {_fehlend}"
for k in EINGRIFF_KONTROLLEN:
    v = next(v for v in VARIANTEN if v.text == k)
    assert len(v.zerlegung) == len(ZIEL_ZERLEGUNG), (
        f"Eingriffskontrolle {k} ist nicht strukturgleich: {list(v.zerlegung)}")

print(f"Varianten gesamt: {len(VARIANTEN)}")
print(f"strukturgleiche Buchstabenkontrollen: {len(BRAUCHBAR)}")
print(f"verworfen: {VERWORFEN}")
for v in VARIANTEN:
    if v.rolle != "buchstabenkontrolle":
        print(f"  {v.rolle:32s} {v.text:6s} {list(v.zerlegung)} {list(v.token_ids)}")

assert not maengel, f"STRUKTURPRUEFUNG GESCHEITERT: {maengel}"
print("\nStrukturpruefung bestanden.")

## 3 · Modell laden und Identität prüfen

`pythia-1b` und `pythia-1.4b` haben dieselbe Breite (2048) und dieselbe Zwischenschicht
(8192). Eine Verwechslung liefert gleich dimensionierte Aktivierungen und **scheitert
still**. Deshalb wird die Blockzahl explizit geprüft.

In [ ]:
dtype = torch.float32 if PRAEZISION == "float32" else torch.bfloat16
model = AutoModelForCausalLM.from_pretrained(MODELL_ID, revision=REVISION, torch_dtype=dtype)
model.eval()
GERAET = "cuda" if torch.cuda.is_available() else "cpu"
model.to(GERAET)

cfg = model.config
BLOECKE = model.gpt_neox.layers
N_BLOECKE = len(BLOECKE)

ERWARTET = {"num_hidden_layers": 24, "hidden_size": 2048, "num_attention_heads": 16,
            "intermediate_size": 8192, "use_parallel_residual": True}
ist = {k: getattr(cfg, k) for k in ERWARTET}
print("Architektur:", ist)
assert ist == ERWARTET, f"MODELLVERWECHSLUNG? erwartet {ERWARTET}, gefunden {ist}"
assert N_BLOECKE == 24
print(f"\nGeladen: {MODELL_ID} @ {REVISION}, {PRAEZISION}, {N_BLOECKE} Bloecke, Geraet {GERAET}")

## 4 · Messgröße

In [ ]:
MARKER_IDS = {}
for feld, woerter in MESSVOKABULAR.items():
    ids = []
    for w in woerter:
        t = tokenizer.encode(w)
        assert len(t) == 1, f"{w!r} ist kein Einzeltoken: {tokenizer.convert_ids_to_tokens(t)}"
        ids.append(t[0])
    MARKER_IDS[feld] = ids
    print(f"  {feld:10s} {ids}  {[tokenizer.convert_ids_to_tokens([i])[0] for i in ids]}")

# Gegen die vorab abgelesene Tabelle. Das faengt genau den Fall, den die
# Einzeltokenpruefung nicht faengt: richtige Tokenzahl, andere Tokenisierung.
_abweichend = {f: (tuple(MARKER_IDS[f]), MESSVOKABULAR_IDS[f])
               for f in MESSVOKABULAR if tuple(MARKER_IDS[f]) != MESSVOKABULAR_IDS[f]}
assert not _abweichend, f"TOKENISIERUNG WEICHT AB: {_abweichend}"
print("\nToken-IDs stimmen mit der vorab abgelesenen Tabelle ueberein.")

ZAEHLER = {"forward": 0}

@torch.no_grad()
def messe(prompt, hooks=()):
    """Ein Vorwaertslauf. Gibt je Wortfeld die Log-Wahrscheinlichkeitsmasse in nats."""
    ids = tokenizer.encode(prompt, return_tensors="pt").to(GERAET)
    griffe = [ziel.register_forward_hook(fn) if art == "post" else
              ziel.register_forward_pre_hook(fn)
              for ziel, fn, art in hooks]
    try:
        out = model(ids)
        ZAEHLER["forward"] += 1
    finally:
        for g in griffe:
            g.remove()
    logprobs = torch.log_softmax(out.logits[0, -1, :].float(), dim=-1)
    werte = {feld: torch.logsumexp(logprobs[idx], dim=0).item()
             for feld, idx in MARKER_IDS.items()}
    werte["_n_tokens"] = ids.shape[1]
    return werte

## 5 · Grundlinien

30 Varianten × 24 Schablonen = 720 Vorwärtsläufe.

In [ ]:
import csv

grundlinien = {}
zeilen = []
t0 = time.time()
for vi, va in enumerate(VARIANTEN):
    for si, sch in enumerate(SCHABLONEN):
        prompt = sch.format(ziel=va.text)
        w = messe(prompt)
        grundlinien[(va.text, si)] = w
        zeilen.append({"variante": va.text, "rolle": va.rolle, "schablone": si,
                       "n_tokens": w["_n_tokens"],
                       **{f: round(w[f], 6) for f in MESSVOKABULAR}})
    if vi % 10 == 0:
        print(f"  {vi+1}/{len(VARIANTEN)} Varianten, {ZAEHLER['forward']} Laeufe, {time.time()-t0:.0f}s")

with open(f"{AUSGABE}/baselines.csv", "w", newline="") as f:
    w_ = csv.DictWriter(f, fieldnames=list(zeilen[0]))
    w_.writeheader(); w_.writerows(zeilen)

RATE_MS = 1000 * (time.time() - t0) / max(ZAEHLER["forward"], 1)
print(f"\nGrundlinien fertig: {ZAEHLER['forward']} Laeufe, {RATE_MS:.1f} ms je Lauf")

In [ ]:
import statistics

def feldmittel(text, feld="bewegung"):
    return statistics.fmean(grundlinien[(text, si)][feld] for si in range(len(SCHABLONEN)))

# Streuung zwischen Schablonen - die Groesse, an der die Trennschaerfe haengt.
def feldstreuung(text, feld="bewegung"):
    return statistics.pstdev([grundlinien[(text, si)][feld] for si in range(len(SCHABLONEN))])

SIGMA_SCHABLONEN = statistics.fmean([feldstreuung(v.text) for v in VARIANTEN])
print(f"mittlere Streuung zwischen Schablonen: {SIGMA_SCHABLONEN:.4f} nats")
if SIGMA_SCHABLONEN > 0.8:
    print("  ACHTUNG: ueber 0.8 nats - der Lauf gilt laut Vorregistrierung als unterbestimmt.")

print(f"\n{'Variante':8s} {'Rolle':32s} {'Bewegung':>9s} {'Tastatur':>9s} {'Gegenfeld':>10s}")
for v in sorted(VARIANTEN, key=lambda v: -feldmittel(v.text)):
    print(f"  {v.text:6s} {v.rolle:32s} {feldmittel(v.text):9.4f} "
          f"{feldmittel(v.text,'tastatur'):9.4f} {feldmittel(v.text,'gegenfeld'):10.4f}")

## 6 · Qualitätstore

Zwei Tore, die den Lauf abbrechen, wenn sie reißen — mit **verschiedenen** Maßstäben,
und der Unterschied ist der Punkt.

Das **Selbstpatch-Tor** ersetzt eine Aktivierung durch sich selbst. Das ist eine echte
Identität: derselbe Tensor, dieselbe Stelle, derselbe Vorwärtslauf. Es muss **exakt
`0.0`** ergeben, Bit für Bit. Eine Toleranz ließe hier genau den Fehler durch, den das
Tor fangen soll — einen Eingriff, der an der falschen Position sitzt.

Das **Residualtor** prüft dagegen eine *Rekonstruktion*: ob `resid_in + attn + mlp`
den Blockausgang ergibt. Diese Summe läuft in anderer Reihenfolge als im Block selbst,
und FP32 rundet dabei anders — im Probelauf 7.5e-09. Ein Tor auf null hätte einen
völlig korrekten Lauf abgebrochen. Gemessen wird deshalb **relativ** zum Betrag der
Aktivierung, mit Schranke `1e-4`: eng genug, dass eine strukturell falsche Zerlegung
durchfällt, denn die läge um Größenordnungen darüber.

In [ ]:
def block_teile(block):
    """Findet die Hook-Punkte fuer Attention und MLP, robust ueber Versionen."""
    attn = getattr(block, "post_attention_dropout", None) or block.attention
    mlp  = getattr(block, "post_mlp_dropout", None) or block.mlp
    return attn, mlp


@torch.no_grad()
def residual_probe(prompt, schicht):
    """Prueft resid_out = resid_in + attn + mlp (paralleles Residuum)."""
    block = BLOECKE[schicht]
    attn_m, mlp_m = block_teile(block)
    speicher = {}

    def vor(mod, args):
        speicher["resid_in"] = args[0][:, -1, :].detach().clone()

    def nach_block(mod, args, output):
        hs = output[0] if isinstance(output, tuple) else output
        speicher["resid_out"] = hs[:, -1, :].detach().clone()

    def nach_attn(mod, args, output):
        hs = output[0] if isinstance(output, tuple) else output
        speicher["attn"] = hs[:, -1, :].detach().clone()

    def nach_mlp(mod, args, output):
        hs = output[0] if isinstance(output, tuple) else output
        speicher["mlp"] = hs[:, -1, :].detach().clone()

    messe(prompt, hooks=[(block, vor, "pre"), (block, nach_block, "post"),
                         (attn_m, nach_attn, "post"), (mlp_m, nach_mlp, "post")])
    rekonstruiert = speicher["resid_in"] + speicher["attn"] + speicher["mlp"]
    abs_fehler = (rekonstruiert - speicher["resid_out"]).abs().max().item()
    # Am Betrag der Aktivierung gemessen: nur so ist die Zahl ueber Schichten
    # hinweg vergleichbar, deren Aktivierungen um Groessenordnungen wachsen.
    skala = speicher["resid_out"].abs().max().item()
    return abs_fehler, skala


probe_prompt = SCHABLONEN[0].format(ziel=ZIEL)
residual_fehler = []
for l in range(N_BLOECKE):
    abs_fehler, skala = residual_probe(probe_prompt, l)
    residual_fehler.append({"schicht": l, "max_abs_fehler": abs_fehler, "skala": skala,
                            "rel_fehler": abs_fehler / skala if skala > 0 else float("inf")})
MAX_RESIDUAL_FEHLER     = max(r["max_abs_fehler"] for r in residual_fehler)
MAX_RESIDUAL_FEHLER_REL = max(r["rel_fehler"] for r in residual_fehler)
with open(f"{AUSGABE}/parallel_residual_checks.csv", "w", newline="") as f:
    w_ = csv.DictWriter(f, fieldnames=["schicht", "max_abs_fehler", "skala", "rel_fehler"])
    w_.writeheader(); w_.writerows(residual_fehler)
print(f"Residualtor ueber {N_BLOECKE} Bloecke: groesster Fehler absolut "
      f"{MAX_RESIDUAL_FEHLER:.3e}, relativ {MAX_RESIDUAL_FEHLER_REL:.3e} "
      f"(Schranke {HOECHSTER_RESIDUALFEHLER_REL:.0e})")
assert MAX_RESIDUAL_FEHLER_REL <= HOECHSTER_RESIDUALFEHLER_REL, (
    f"RESIDUALTOR GERISSEN ({MAX_RESIDUAL_FEHLER_REL:.3e}) - Lauf abgebrochen")

In [ ]:
@torch.no_grad()
def erfasse_residual(prompt, schicht):
    block = BLOECKE[schicht]
    speicher = {}
    def nach(mod, args, output):
        hs = output[0] if isinstance(output, tuple) else output
        speicher["v"] = hs[:, -1, :].detach().clone()
    messe(prompt, hooks=[(block, nach, "post")])
    return speicher["v"]


def pflanz_hook(vektor):
    def nach(mod, args, output):
        if isinstance(output, tuple):
            hs = output[0].clone(); hs[:, -1, :] = vektor
            return (hs,) + output[1:]
        hs = output.clone(); hs[:, -1, :] = vektor
        return hs
    return nach


@torch.no_grad()
def patche(prompt, schicht, vektor):
    return messe(prompt, hooks=[(BLOECKE[schicht], pflanz_hook(vektor), "post")])


# Selbstpatch: jede Aktivierung durch sich selbst ersetzen. Der Wert muss exakt
# demselben Ergebnis entsprechen wie ohne Eingriff.
selbst_zeilen = []
t0 = time.time()
for name in EINGRIFF_KONTROLLEN:
    for si, sch in enumerate(SCHABLONEN):
        prompt = sch.format(ziel=name)
        for l in range(N_BLOECKE):
            v = erfasse_residual(prompt, l)
            nach = patche(prompt, l, v)
            fehler = max(abs(nach[f] - grundlinien[(name, si)][f]) for f in MESSVOKABULAR)
            selbst_zeilen.append({"variante": name, "schablone": si, "schicht": l,
                                  "max_abs_fehler": fehler})
    print(f"  {name} fertig, {ZAEHLER['forward']} Laeufe, {time.time()-t0:.0f}s")

MAX_SELBSTPATCH_FEHLER = max(z["max_abs_fehler"] for z in selbst_zeilen)
with open(f"{AUSGABE}/self_patch_checks.csv", "w", newline="") as f:
    w_ = csv.DictWriter(f, fieldnames=list(selbst_zeilen[0]))
    w_.writeheader(); w_.writerows(selbst_zeilen)
print(f"\nSelbstpatch-Tor: groesster Fehler = {MAX_SELBSTPATCH_FEHLER:.3e} "
      f"ueber {len(selbst_zeilen)} Pruefungen")
assert MAX_SELBSTPATCH_FEHLER == 0.0, "SELBSTPATCH-TOR GERISSEN - Lauf abgebrochen"

## 7 · Transfers

Fünf Paare × 24 Schablonen × 24 Blöcke = **2 880 Transferzeilen**. Jede Zeile erfasst
zwei Aktivierungen und setzt zwei Eingriffe — Zielaktivierung in den Kontrollkontext
und umgekehrt — also **11 520 Vorwärtsläufe**.

Die Gegenrichtung ist nicht Beiwerk: Nur sie trennt einen tokenspezifischen Effekt von
einem, der bloß an der Position hängt. Trägt der Eingriff in beide Richtungen gleich
gut, liegt es an der Stelle im Satz und nicht an `WASD`.

In [ ]:
transfer_zeilen = []
t0 = time.time()
for kontrolle in EINGRIFF_KONTROLLEN:
    for si, sch in enumerate(SCHABLONEN):
        p_ziel = sch.format(ziel=ZIEL)
        p_ktrl = sch.format(ziel=kontrolle)
        b_ziel = grundlinien[(ZIEL, si)]["bewegung"]
        b_ktrl = grundlinien[(kontrolle, si)]["bewegung"]
        abstand = b_ziel - b_ktrl
        for l in range(N_BLOECKE):
            v_ziel = erfasse_residual(p_ziel, l)
            v_ktrl = erfasse_residual(p_ktrl, l)
            # Richtung 1: Zielaktivierung in den Kontrollkontext
            nach_hin = patche(p_ktrl, l, v_ziel)["bewegung"]
            # Richtung 2: Kontrollaktivierung in den Zielkontext
            nach_her = patche(p_ziel, l, v_ktrl)["bewegung"]
            brauchbar = abs(abstand) >= MINDEST_ABSTAND_NATS
            transfer_zeilen.append({
                "kontrolle": kontrolle, "schablone": si, "schicht": l,
                "baseline_ziel": round(b_ziel, 6), "baseline_kontrolle": round(b_ktrl, 6),
                "abstand": round(abstand, 6),
                "gepatcht_ziel_in_kontrolle": round(nach_hin, 6),
                "gepatcht_kontrolle_in_ziel": round(nach_her, 6),
                "wiederherstellung_ziel": round((nach_hin - b_ktrl) / abstand, 6) if brauchbar else None,
                "leck_kontrolle": round((b_ziel - nach_her) / abstand, 6) if brauchbar else None,
                "abstand_zu_klein": not brauchbar,
            })
    print(f"  Paar {ZIEL}/{kontrolle} fertig, {ZAEHLER['forward']} Laeufe, {time.time()-t0:.0f}s")

with open(f"{AUSGABE}/patch_responses.csv", "w", newline="") as f:
    w_ = csv.DictWriter(f, fieldnames=list(transfer_zeilen[0]))
    w_.writeheader(); w_.writerows(transfer_zeilen)

ANTEIL_ZU_KLEIN = sum(z["abstand_zu_klein"] for z in transfer_zeilen) / len(transfer_zeilen)
print(f"\nTransfers: {len(transfer_zeilen)} Zeilen, "
      f"{ANTEIL_ZU_KLEIN:.1%} mit zu kleinem Abstand")
assert ANTEIL_ZU_KLEIN <= HOECHSTER_ANTEIL_ZU_KLEINER_ABSTAND, (
    f"ZU VIELE UNBRAUCHBARE NENNER ({ANTEIL_ZU_KLEIN:.1%}) - Lauf abgebrochen")

## 8 · Auswertung nach der vorregistrierten Regel

In [ ]:
# --- Beobachtungsteil -------------------------------------------------------
IN_DER_REGEL = [v for v in VARIANTEN if v.rolle not in ("ziel", ROLLE_GETRENNT)]
GETRENNT     = [v for v in VARIANTEN if v.rolle == ROLLE_GETRENNT]

ziel_wert   = feldmittel(ZIEL)
kontrollen  = sorted(((feldmittel(v.text), v.text) for v in IN_DER_REGEL), reverse=True)
beste_wert, beste_name = kontrollen[0]
vorsprung = ziel_wert - beste_wert

werte = sorted(w for w, _ in kontrollen)
stelle = KONTROLL_QUANTIL * (len(werte) - 1)
unten = int(stelle); oben = min(unten + 1, len(werte) - 1)
quantil = werte[unten] + (stelle - unten) * (werte[oben] - werte[unten])
vorsprung_quantil = ziel_wert - quantil

beobachtung_haelt = vorsprung >= MINDEST_VORSPRUNG
beobachtung_quantil = vorsprung_quantil >= MINDEST_VORSPRUNG

print(f"Ziel {ZIEL}: {ziel_wert:.4f} nats")
print(f"beste Kontrolle {beste_name}: {beste_wert:.4f}  ->  Vorsprung {vorsprung:+.4f}")
print(f"{int(KONTROLL_QUANTIL*100)}%-Quantil der Kontrollen: {quantil:.4f}  ->  Vorsprung {vorsprung_quantil:+.4f}")
print(f"\nBeobachtungsteil (Hauptregel):     {beobachtung_haelt}")
print(f"Beobachtungsteil (Quantilsfassung): {beobachtung_quantil}")
for v in GETRENNT:
    print(f"getrennt berichtet: {v.text} = {feldmittel(v.text):.4f} nats (nicht in der Regel)")

In [ ]:
# --- Kausalteil -------------------------------------------------------------
# Je Block ueber Schablonen und Paare mitteln, dann den laengsten Lauf benachbarter
# Bloecke suchen, an denen beide Bedingungen gleichzeitig halten.
je_block = {}
for l in range(N_BLOECKE):
    zs = [z for z in transfer_zeilen if z["schicht"] == l and not z["abstand_zu_klein"]]
    if not zs:
        je_block[l] = (float("nan"), float("nan")); continue
    je_block[l] = (statistics.fmean(z["wiederherstellung_ziel"] for z in zs),
                   statistics.fmean(z["leck_kontrolle"] for z in zs))

laeufe, aktuell = [], 0
for l in range(N_BLOECKE):
    wh, leck = je_block[l]
    traegt = (wh == wh) and wh >= MINDEST_KAUSALANTEIL and leck <= HOECHSTES_LECK
    aktuell = aktuell + 1 if traegt else 0
    laeufe.append(aktuell)
LAENGSTER_LAUF = max(laeufe) if laeufe else 0
kausal_haelt = LAENGSTER_LAUF >= MINDEST_SCHICHTEN

print(f"{'Block':>5s} {'Wiederherstellung':>18s} {'Leck':>8s}  traegt")
for l in range(N_BLOECKE):
    wh, leck = je_block[l]
    traegt = (wh == wh) and wh >= MINDEST_KAUSALANTEIL and leck <= HOECHSTES_LECK
    print(f"  {l:3d} {wh:18.4f} {leck:8.4f}  {'ja' if traegt else ''}")
print(f"\nlaengster Lauf benachbarter Bloecke: {LAENGSTER_LAUF}")
print(f"Kausalteil: {kausal_haelt}")

BESTAETIGT = bool(beobachtung_haelt and kausal_haelt)
print(f"\n{'='*52}\nwasd_traeger_bestaetigt = {BESTAETIGT}\n{'='*52}")

## 9 · Ausgabedateien

In [ ]:
ZUSAMMENFASSUNG = {
    "modell": MODELL_ID, "revision": REVISION, "praezision": PRAEZISION,
    "batchgroesse": BATCHGROESSE, "geraet": GERAET,
    "n_varianten": len(VARIANTEN), "n_strukturgleiche_kontrollen": len(BRAUCHBAR),
    "verworfen": VERWORFEN, "n_schablonen": len(SCHABLONEN),
    "sigma_zwischen_schablonen": round(SIGMA_SCHABLONEN, 6),
    "unterbestimmt": SIGMA_SCHABLONEN > 0.8,
    "ziel_bewegung_nats": round(ziel_wert, 6),
    "beste_kontrolle": beste_name,
    "beste_kontrolle_nats": round(beste_wert, 6),
    "vorsprung_nats": round(vorsprung, 6),
    "vorsprung_gegen_quantil_nats": round(vorsprung_quantil, 6),
    "beobachtungsteil": beobachtung_haelt,
    "beobachtungsteil_quantilsfassung": beobachtung_quantil,
    "getrennt_berichtet": {v.text: round(feldmittel(v.text), 6) for v in GETRENNT},
    "laengster_kausallauf": LAENGSTER_LAUF,
    "kausalteil": kausal_haelt,
    "wasd_traeger_bestaetigt": BESTAETIGT,
    "regel": {"mindest_vorsprung_nats": MINDEST_VORSPRUNG,
              "kontroll_quantil": KONTROLL_QUANTIL,
              "mindest_kausalanteil": MINDEST_KAUSALANTEIL,
              "hoechstes_leck": HOECHSTES_LECK,
              "mindest_schichten": MINDEST_SCHICHTEN},
}
json.dump(ZUSAMMENFASSUNG, open(f"{AUSGABE}/FINAL_SUMMARY.json", "w"), indent=2, ensure_ascii=False)

QA = {
    "vorflug": VORFLUG,
    "architektur": ist,
    "max_selbstpatch_fehler": MAX_SELBSTPATCH_FEHLER,
    "max_residual_fehler": MAX_RESIDUAL_FEHLER,
    "max_residual_fehler_relativ": MAX_RESIDUAL_FEHLER_REL,
    "schranke_residual_relativ": HOECHSTER_RESIDUALFEHLER_REL,
    "anteil_abstand_zu_klein": round(ANTEIL_ZU_KLEIN, 6),
    "strukturpruefung_bestanden": True,
    "tokenizer_sha256": TOKENIZER_SHA256,
}
json.dump(QA, open(f"{AUSGABE}/QA.json", "w"), indent=2, ensure_ascii=False)

NACHWEIS = {
    "forward_laeufe_gesamt": ZAEHLER["forward"],
    "grundlinien": len(VARIANTEN) * len(SCHABLONEN),
    "selbstpatch_pruefungen": len(selbst_zeilen),
    "transferzeilen": len(transfer_zeilen),
    "ms_je_lauf": round(RATE_MS, 2),
    "hinweis": "Die Wortfelder sind lexikalische Sonden, keine validierten Klassifikatoren.",
}
json.dump(NACHWEIS, open(f"{AUSGABE}/measurement_evidence.json", "w"), indent=2, ensure_ascii=False)

bericht = f"""# WASD-Traegerexperiment - Messlauf

- Modell: `{MODELL_ID}` @ `{REVISION}`, {PRAEZISION}, Batchgroesse {BATCHGROESSE}
- Varianten: {len(VARIANTEN)}, davon {len(BRAUCHBAR)} strukturgleiche Buchstabenkontrollen
- Verworfen (andere Zerlegung): {VERWORFEN}
- Schablonen: {len(SCHABLONEN)}
- Vorwaertslaeufe: {ZAEHLER['forward']} ({RATE_MS:.1f} ms je Lauf)

## Qualitaetstore
- Selbstpatch, groesster Fehler: {MAX_SELBSTPATCH_FEHLER:.3e}
- Paralleles Residuum, groesster Fehler: {MAX_RESIDUAL_FEHLER:.3e} absolut, {MAX_RESIDUAL_FEHLER_REL:.3e} relativ (Schranke {HOECHSTER_RESIDUALFEHLER_REL:.0e})
- Transferzeilen mit zu kleinem Abstand: {ANTEIL_ZU_KLEIN:.1%}
- Streuung zwischen Schablonen: {SIGMA_SCHABLONEN:.4f} nats{' (UNTERBESTIMMT)' if SIGMA_SCHABLONEN > 0.8 else ''}

## Beobachtungsteil
- Ziel `{ZIEL}`: {ziel_wert:.4f} nats
- beste Kontrolle `{beste_name}`: {beste_wert:.4f} nats
- Vorsprung: {vorsprung:+.4f} nats (Schwelle {MINDEST_VORSPRUNG})
- Vorsprung gegen das {int(KONTROLL_QUANTIL*100)}%-Quantil: {vorsprung_quantil:+.4f} nats
- **Hauptregel: {beobachtung_haelt}**, Quantilsfassung: {beobachtung_quantil}

## Kausalteil
- laengster Lauf benachbarter Bloecke: {LAENGSTER_LAUF} (verlangt {MINDEST_SCHICHTEN})
- **Kausalteil: {kausal_haelt}**

## Gesamturteil
`wasd_traeger_bestaetigt = {BESTAETIGT}`

Beide Teile muessen halten. Ein einzelner positiver Teil zaehlt nicht.

## Grenzen
- Gemessen wird an eigens gebauten Saetzen, nicht am Textfenster State 60482.
- Die Wortfelder sind lexikalische Sonden, keine validierten Klassifikatoren.
- Der Eingriff sitzt am letzten Token. Nur das Leck der Kontrollquelle trennt einen
  tokenspezifischen von einem rein positionsbedingten Effekt.
- `ESDF` geht nicht in die Regel ein: andere Zerlegung, deutlich seltener im Korpus.
"""
open(f"{AUSGABE}/FINAL_REPORT.md", "w").write(bericht)
json.dump({"abgeschlossen": time.strftime("%Y-%m-%d %H:%M:%S")},
          open(f"{AUSGABE}/COMPLETE", "w"), indent=2)

print(bericht)
print("Geschrieben nach", AUSGABE, ":", sorted(os.listdir(AUSGABE)))

## 10 · Ergebnisse sichern

`AUSGABE` liegt im Dateisystem der Colab-Laufzeit. Das ist **flüchtig**: trennt die
Laufzeit, sind alle Zahlen dieses Laufs verloren, und die achtzehntausend
Vorwärtsläufe müssten wiederholt werden.

Diese Zelle kopiert den Ordner nach Drive. Sie läuft außerhalb von Colab wirkungslos
durch, damit das Notebook auch anderswo bis zum Ende durchläuft.

In [ ]:
# Sichern, bevor die Laufzeit trennt. Ohne Colab passiert hier nichts.
import glob, shutil

def sichere_nach_drive(quelle=AUSGABE, unterordner="WASD_Traeger_Messlauf"):
    try:
        from google.colab import drive  # type: ignore
    except ImportError:
        print("Kein Colab - nichts zu sichern. Ergebnisse liegen in", os.path.abspath(quelle))
        return None
    drive.mount("/content/drive")
    # Den bestehenden Ergebnisordner des Projekts suchen, statt einen Pfad zu raten.
    treffer = (glob.glob("/content/drive/MyDrive/Colab_Pythia_Results")
               + glob.glob("/content/drive/Shareddrives/*/Colab_Pythia_Results"))
    basis = treffer[0] if treffer else "/content/drive/MyDrive"
    if not treffer:
        print("Colab_Pythia_Results nicht gefunden - sichere nach MyDrive.")
    ziel = f"{basis}/{unterordner}/{time.strftime('%Y%m%d_%H%M%S')}"
    shutil.copytree(quelle, ziel)
    return ziel


GESICHERT = sichere_nach_drive()
if GESICHERT:
    print("Gesichert nach:", GESICHERT)
    for name in sorted(os.listdir(GESICHERT)):
        print(f"  {name:32s} {os.path.getsize(os.path.join(GESICHERT, name)):8d} B")